micrograd = mini librairie eg. PyTorch 

backpropagation = rétropropagation du gradient -> ajusté les poids/paramètres / méthode : récursive chain rule
		  
		  

In [44]:
import math
import random

In [ ]:
class Value:

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self._prev = set(_children)
        self._op = _op
        self.grad = 0
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data = {self.data})"

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)     # L'équivalent d'un dynamic cast
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out
    
    def __radd__(self, other):
        return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out =  Value(self.data*other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        
        out._backward = _backward

        return out
    
    def __rmul__(self, other):      # interverti automatiquement self et other lors de l'appel
        return self * other
    
    def __neg__(self):
        return self*(-1)

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)
    
    # fonction puissance valide que si self = Value et si self.data > 0
    def __pow__(self, other):
        assert isinstance(other, (int, float))
        out = Value(self.data ** other, (self,), '**')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad

        out._backward = _backward
        return out
    
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * (other**(-1))

    def tanh(self):
        x = self.data
        th = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
        out = Value(th, (self,), "tanh")

        def _backward():
            self.grad += (1 - th**2) * out.grad
        
        out._backward = _backward

        return out

    def backward(self):

        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1
        for node in reversed(topo):
            node._backward()


In [144]:
# inputs
x1 = Value(2)
x2 = Value(3)

#weights
w1 = Value(-3)
w2 = Value(1)

# biais of the neuron
b = Value(6.7)

# operations of the neuron
x1w1 = x1*w1
x2w2 = x2*w2
som = x1w1 + x2w2
somb = som + b
l = somb.tanh()


In [145]:
l.backward()
print(l.grad)
print(somb.grad)
print(som.grad)
print(x2w2.grad)
print(x1w1.grad)
print(x1.grad)
print(x2.grad)
print(w1.grad)
print(w2.grad)

1
0.002442024743370408
0.002442024743370408
0.002442024743370408
0.002442024743370408
-0.007326074230111224
0.002442024743370408
0.004884049486740816
0.007326074230111224


In [146]:
class Neuron:

    # Nin = Nombre d'inputs du neurone
    def __init__(self, Nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(Nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)        # w.x + b   # Les opérateurs add et mul de la classe Value sont utilisés implicitement
        out = act.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

In [147]:
x = [2, 3]
n = Neuron(2)
n(x)

Value(data = -0.9969246868905949)

In [148]:
class Layer:

    # Nin = Nombre d'inputs / Nout = Nombre de neurones dans la layer (nombre de sorties) 
    def __init__(self, Nin, Nout):
        self.neurons = [Neuron(Nin) for _ in range(Nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs      # résultat du réseau de neurones sous forme d'une valeur (et non d'une liste)
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

In [149]:
x = [2, 3]
l = Layer(2, 3)
l(x)

[Value(data = -0.96748517455092),
 Value(data = -0.3704780618303362),
 Value(data = 0.8806983502206392)]

In [150]:
class MLP:

    # Nin = Nombre d'inputs / Nouts = Liste du nombre d'output de chaque layer
    def __init__(self, Nin, Nouts):
        sz = [Nin] + Nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(Nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x 
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [205]:
x = [2, 3, -1]
mlp = MLP(3, [4, 4, 1])
mlp(x)

Value(data = 0.011822570003563466)

In [ ]:
# Liste d'inputs (x)
xs = [
    [2, 3, -1],
    [3, -1, 0.5],
    [0.5, 1, 1],
    [1, 1, -1],
]

# Desired targets
ys = [1, -1, -1, 1]

ypred = [mlp(x) for x in xs]

loss = sum((ytgt-yout)**2 for ytgt, yout in zip(ys, ypred))
print(loss)

Niterations = 0

while loss.data > 0.01:
    
    Niterations +=1

    #back pass
    loss.backward()

    for p in mlp.parameters():
        
        p.data -= 0.01*p.grad   # implémentation du learning rate
        p.grad = 0
    
    #forward pass
    ypred = [mlp(x) for x in xs]
    loss = sum((ytgt-yout)**2 for ytgt, yout in zip(ys, ypred))

print(loss)
print(Niterations)

Value(data = 5.751075009823808)
Value(data = 0.009999888709079953)
318
